# The N-Queen

In [31]:
import sys
import time
from ortools.sat.python import cp_model

In [32]:
model = cp_model.CpModel()
board_size = 4

In [33]:
# There are `board_size` number of variables, one for a queen in each column q[j] = i
# of the board. The value of each variable is the row that the queen is in.
queens = [model.new_int_var(0, board_size - 1, f"x_{i}") for i in range(board_size)]
print(queens)
print(queens[1]+1)

[x_0(0..3), x_1(0..3), x_2(0..3), x_3(0..3)]
(x_1 + 1)


In [34]:
# All rows must be different.
model.add_all_different(queens)

# No two queens can be on the same diagonal.
model.add_all_different(queens[i] + i for i in range(board_size))
model.add_all_different(queens[i] - i for i in range(board_size))

In [35]:
class NQueenSolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, queens: list[cp_model.IntVar]):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__queens = queens
        self.__solution_count = 0
        self.__start_time = time.time()

    @property
    def solution_count(self) -> int:
        return self.__solution_count

    def on_solution_callback(self):
        current_time = time.time()
        print(
            f"Solution {self.__solution_count}, "
            f"time = {current_time - self.__start_time} s"
        )
        self.__solution_count += 1

        all_queens = range(len(self.__queens))
        for i in all_queens:
            for j in all_queens:
                if self.value(self.__queens[j]) == i:
                    # There is a queen in column j, row i.
                    print("Q", end=" ")
                else:
                    print("_", end=" ")
            print()
        print()

In [37]:
solver = cp_model.CpSolver()
# solution_printer = NQueenSolutionPrinter(queens)
# solver.parameters.enumerate_all_solutions = True
# solver.solve(model, solution_printer)
status = solver.solve(model)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"成功找到 {board_size} 皇后问题的一个解！\n")
        
    # 打印可视化棋盘
    for i in range(board_size):
        # 获取第 i 行皇后所在的列号
        queen_col = solver.Value(queens[i])
        
        # 绘制当前行
        row_str = ""
        for j in range(board_size):
            if j == queen_col:
                row_str += " ♛ "  # 皇后的位置
            else:
                row_str += " . "   # 空白位置
        print(row_str)
else:
    print("没有找到解决方案。")

成功找到 4 皇后问题的一个解！

 .  .  ♛  . 
 ♛  .  .  . 
 .  .  .  ♛ 
 .  ♛  .  . 


# Setting a time limit for the solver

In [42]:
"""Solves a problem with a time limit."""

from ortools.sat.python import cp_model


def solve_with_time_limit_sample_sat():
    """Minimal CP-SAT example to showcase calling the solver."""
    # Creates the model.
    model = cp_model.CpModel()
    # Creates the variables.
    num_vals = 3
    x = model.new_int_var(0, num_vals - 1, "x")
    y = model.new_int_var(0, num_vals - 1, "y")
    z = model.new_int_var(0, num_vals - 1, "z")
    # Adds an all-different constraint.
    model.add(x != y)

    # Creates a solver and solves the model.
    solver = cp_model.CpSolver()

    # Sets a time limit of 10 seconds.
    solver.parameters.max_time_in_seconds = 10.0

    status = solver.solve(model)

    if status == cp_model.FEASIBLE:
        print(f"x = {solver.value(x)}")
        print(f"y = {solver.value(y)}")
        print(f"z = {solver.value(z)}")


solve_with_time_limit_sample_sat()

# The Job Shop Problem
![车间作业调度问题（JSSP）](./pics/schedule1.png)  

In [1]:
import collections
from ortools.sat.python import cp_model

In [ ]:
# 1. 业务数据定义
# jobs[i] 表示第 i 个工件。内部列表是按先后顺序执行的工序：(机器ID, 耗时)
jobs_data = [
    [(0, 3), (1, 2), (2, 2)],  # 工件 0: 先在机器0做3小时，再去机器1做2小时，再去机器2做2小时
    [(0, 2), (2, 1), (1, 4)],  # 工件 1: 先在机器0做2小时，再去机器2做1小时，再去机器1做4小时
    [(1, 4), (2, 3)]           # 工件 2: 先在机器1做4小时，再去机器2做3小时 (这个工件只有2道工序)
]

num_machines = 3

# 计算一个绝对安全的最大时间界限 (Horizon)
# 最坏的情况就是所有任务串行执行，总时间不会超过所有耗时之和
horizon = sum(task[1] for job in jobs_data for task in job)

In [44]:
# 2. 创建模型
model = cp_model.CpModel()

In [45]:
# 存储变量的数据字典
# all_tasks[job_id, task_id] = { 'start': 变量, 'end': 变量, 'interval': 区间变量 }
all_tasks = {}

# machine_to_intervals[machine_id] = [区间变量1, 区间变量2, ...]
machine_to_intervals = collections.defaultdict(list)

# 3. 创建变量
for job_id, job in enumerate(jobs_data):
    for task_id, task in enumerate(job):
        machine_id = task[0]
        duration = task[1]
        suffix = f'_j{job_id}_t{task_id}_m{machine_id}'

        # 定义开始时间和结束时间变量 (范围从 0 到 horizon)
        start_var = model.new_int_var(0, horizon, 'start' + suffix)
        end_var = model.new_int_var(0, horizon, 'end' + suffix)
        
        # 🌟 核心：定义区间变量 (Interval Variable)
        interval_var = model.new_interval_var(
            start_var, duration, end_var, 'interval' + suffix)
        
        all_tasks[job_id, task_id] = {
            'start': start_var,
            'end': end_var,
            'interval': interval_var
        }
        # 把区间变量记录到对应的机器下
        machine_to_intervals[machine_id].append(interval_var)

print(all_tasks)
print(machine_to_intervals)

{(0, 0): {'start': start_j0_t0_m0(0..21), 'end': end_j0_t0_m0(0..21), 'interval': interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0)}, (0, 1): {'start': start_j0_t1_m1(0..21), 'end': end_j0_t1_m1(0..21), 'interval': interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1)}, (0, 2): {'start': start_j0_t2_m2(0..21), 'end': end_j0_t2_m2(0..21), 'interval': interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2)}, (1, 0): {'start': start_j1_t0_m0(0..21), 'end': end_j1_t0_m0(0..21), 'interval': interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)}, (1, 1): {'start': start_j1_t1_m2(0..21), 'end': end_j1_t1_m2(0..21), 'interval': interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2)}, (1, 2): {'start': start_j1_t2_m1(0..21), 'end': end_j1_t2_m1(0..21), 'interval': interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1)}, (2, 0): {'start': start_j2_t0_m1(0..21), 'end': end_j2_t0_m1(0..21), 

```python
# all_tasks
{
    (0, 0): {'start': start_j0_t0_m0(0..21), 'end': end_j0_t0_m0(0..21), 'interval': interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0)}, 
    (0, 1): {'start': start_j0_t1_m1(0..21), 'end': end_j0_t1_m1(0..21), 'interval': interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1)}, 
    (0, 2): {'start': start_j0_t2_m2(0..21), 'end': end_j0_t2_m2(0..21), 'interval': interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2)}, 
    (1, 0): {'start': start_j1_t0_m0(0..21), 'end': end_j1_t0_m0(0..21), 'interval': interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)}, 
    (1, 1): {'start': start_j1_t1_m2(0..21), 'end': end_j1_t1_m2(0..21), 'interval': interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2)}, 
    (1, 2): {'start': start_j1_t2_m1(0..21), 'end': end_j1_t2_m1(0..21), 'interval': interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1)}, 
    (2, 0): {'start': start_j2_t0_m1(0..21), 'end': end_j2_t0_m1(0..21), 'interval': interval_j2_t0_m1(start = start_j2_t0_m1, size = 4, end = end_j2_t0_m1)}, 
    (2, 1): {'start': start_j2_t1_m2(0..21), 'end': end_j2_t1_m2(0..21), 'interval': interval_j2_t1_m2(start = start_j2_t1_m2, size = 3, end = end_j2_t1_m2)}
}

# machine_to_intervals
defaultdict(<class 'list'>, 
    {
        0: [interval_j0_t0_m0(start = start_j0_t0_m0, size = 3, end = end_j0_t0_m0), interval_j1_t0_m0(start = start_j1_t0_m0, size = 2, end = end_j1_t0_m0)], 
        1: [interval_j0_t1_m1(start = start_j0_t1_m1, size = 2, end = end_j0_t1_m1), interval_j1_t2_m1(start = start_j1_t2_m1, size = 4, end = end_j1_t2_m1), interval_j2_t0_m1(start = start_j2_t0_m1, size = 4, end = end_j2_t0_m1)], 
        2: [interval_j0_t2_m2(start = start_j0_t2_m2, size = 2, end = end_j0_t2_m2), interval_j1_t1_m2(start = start_j1_t1_m2, size = 1, end = end_j1_t1_m2), interval_j2_t1_m2(start = start_j2_t1_m2, size = 3, end = end_j2_t1_m2)]
    }
)
```

In [46]:
# 4. 添加约束
# 约束 A：同一个工件的工序必须按顺序执行 (后一个工序的开始时间 >= 前一个工序的结束时间)
for job_id, job in enumerate(jobs_data):
    for task_id in range(1, len(job)):
        prev_end = all_tasks[job_id, task_id - 1]['end']
        curr_start = all_tasks[job_id, task_id]['start']
        model.add(curr_start >= prev_end)

# 约束 B：🌟 核心：同一台机器上的任何任务不能在时间上重叠
for machine_id in range(num_machines):
    model.add_no_overlap(machine_to_intervals[machine_id])

In [47]:
# 5. 定义目标函数：最小化总完工时间 (Makespan)
# 总完工时间 >= 所有工件最后一个工序的结束时间
makespan = model.new_int_var(0, horizon, 'makespan')
makespan_ends = []
for job_id, job in enumerate(jobs_data):
    last_task_id = len(job) - 1
    makespan_ends.append(all_tasks[job_id, last_task_id]['end'])
    
model.add_max_equality(makespan, makespan_ends) # makespan = max(所有最后一个工序的end)
model.minimize(makespan)

In [48]:
# 6. 调用求解器
solver = cp_model.CpSolver()
status = solver.Solve(model)

In [50]:
# 7. 打印结果 (以机器维度格式化输出排产计划)
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f'最优总完工时间 (Makespan): {solver.objective_value} 小时\n')

    # 组织输出数据
    assigned_jobs = collections.defaultdict(list)
    for job_id, job in enumerate(jobs_data):
        for task_id, task in enumerate(job):
            machine_id = task[0]
            assigned_jobs[machine_id].append({
                'job': job_id,
                'task': task_id,
                'start': solver.value(all_tasks[job_id, task_id]['start']),
                'duration': task[1]
            })

    # 按机器打印时间表
    for machine_id in range(num_machines):
        # 将该机器上的任务按开始时间排序
        assigned_jobs[machine_id].sort(key=lambda x: x['start'])
        
        schedule_str = f'机器 {machine_id}: '
        for task_dict in assigned_jobs[machine_id]:
            job = task_dict['job']
            task = task_dict['task']
            start = task_dict['start']
            duration = task_dict['duration']
            end = start + duration
            
            schedule_str += f'[工件{job}_工序{task}: {start:>2}点 -> {end:>2}点]  '
        print(schedule_str)
else:
    print('没有找到可行的排产方案。')

最优总完工时间 (Makespan): 11.0 小时

机器 0: [工件1_工序0:  0点 ->  2点]  [工件0_工序0:  2点 ->  5点]  
机器 1: [工件2_工序0:  0点 ->  4点]  [工件0_工序1:  5点 ->  7点]  [工件1_工序2:  7点 -> 11点]  
机器 2: [工件1_工序1:  2点 ->  3点]  [工件2_工序1:  4点 ->  7点]  [工件0_工序2:  7点 ->  9点]  


DAG
负责描述实验步骤依赖
        ↓
RCPSP
把 DAG + 资源约束转成调度优化问题
        ↓
Rolling Horizon Scheduling
每隔一段时间构建一个局部 RCPSP 问题
        ↓
CP-SAT
求解这个局部调度问题，输出可执行计划 -> DAG (给用户输出？)
        ↓
硬件层      


# 直接交换死锁

在实验室自动化（Lab Automation）或多机械臂矩阵中，如果系统控制逻辑是：“机械臂每次运动前，必须判断目标设备（坑位）是否为空”，就会极易引发直接交换死锁 (Direct Exchange Deadlock)。

典型崩溃场景：

- 状态： 孵化器（满，装有板子A），离心机（满，装有板子B）。两者均已完成实验。
- 目标： 板子A需要去离心机，板子B需要去孵化器。
- 死锁原因： 机械臂尝试抓取板子A时，判断目标（离心机）不为空，于是拒绝动作，原地等待；同理，另一侧判断孵化器不为空，也原地等待。
- 本质： 系统试图在没有“中间空地（临时变量）”的情况下，让两个物理实体互换位置。如同两人在狭窄独木桥上相遇，都等对方先退，导致永远僵持。

---

# 破解方案
1. 引入“物理缓存站（Buffer/Hotel）”

在软件工程中，交换两个变量 a 和 b 需要一个临时变量 temp。物理硬件同理。

设计思路： 在工作台面上预留至少一个公共的、空闲的板位（通常称为 Buffer Rack 或 Hotel）。
运行逻辑（中心调度器强行介入）：
- 中央调度器检测到死锁条件成立（双方皆满且互需位置）。
- 拦截常规等待指令，下发**“死锁消解宏指令”**。
- 臂 A 取出孵化器里的板子，但不去离心机，而是放到 Buffer位 上。 (此时：孵化器腾空！)
- 臂 B 看到孵化器空了，取出离心机里的板子，放入孵化器。 (此时：离心机腾空！)
- 臂 A 去 Buffer 位拿回原板子，放入离心机。完成完美交换。
评价： 最安全、最标准、最容易用软件实现的方案(但有空间要求)。

2. 修改软件检测时机（空中暂存法）

如果硬件台面极其紧凑，连一个 Buffer 暂存位都无法增加，则必须打破机械臂“不见空位不提件”的保守逻辑。

设计思路： 允许机械臂在不满足“目标为空”的情况下，先把物料提出来抓在手里。
运行逻辑：
- 调度器强行命令 臂 A 忽略离心机状态，直接把板子从孵化器里提出来，并停留在安全的半空中等待。 (此时：孵化器物理空间被强行释放！)
- 臂 B 按常规逻辑判断：孵化器已空，于是取出离心机的板子放入孵化器。 (离心机空间释放！)
- 悬停在半空的 臂 A 判定离心机已空，顺势降落将板子放入离心机。
⚠️ 硬件风险警告： 此方案必须在软件中划定严苛的 “3D 安全避让区”。必须确保臂 A 在半空悬停时，绝对不会与正在大范围运动的臂 B 发生物理干涉（碰撞）。

3. 华容道算法（N-1 容量限制法则）

如果你玩过经典的“华容道”或“数字拼图”游戏，就知道：要想让方块能移动，盘面上必须始终留有一个空格。

设计思路： 从任务下发的源头（准入控制）杜绝满载。
运行逻辑：
- 假设系统中总共有 N 个能放微孔板的仪器槽位（无额外Buffer）。
- 调度器限制：整个流水线最多只允许同时运行 N−1 块板子。
- 在只有 1个孵化器 + 1个离心机（N=2）的系统中，强制只能同时跑 1 块板子；若要跑 2 块板子，必须接入第 3 台设备（如洗板机）充当天然 Buffer。
评价： 只要系统中永远飘着一个“空位 (Token)”，资源最终都能像拼图一样倒腾开，从数学层面彻底消灭死锁。缺点是牺牲了部分并发效率。